In [ ]:
%pip install catboost

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)
#df = df.drop(columns="", axis=1)

In [ ]:
type(df.columns)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, 'Delivery_Time')

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Dropping rows where target (Delivery_Time) or key features are missing (I guessed the key features, not sure though)
print(f"Before: {df.shape}")
df_clean = df.dropna(subset=['Delivery_Time', 'Traffic_Level', 'Time_of_Day', 'Weather'])
print(f"After dropping missing target and key features: {df_clean.shape}")

In [ ]:
# Decided to fillna the Courier_Experience_yrs column with the mode
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])

check_missing_values(df_clean)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

for col in categorical_cols:
    print(type(df_clean[col]))
    print(f"Encoding column: {col}")
    onehot_encoder = OneHotEncoder(sparse_output=False)
    df_clean[col] = onehot_encoder.fit_transform(pd.DataFrame(df_clean[col]))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:
# From the distribution in part 1, I think it's mostly balanced, so not gonna fool with it

In [ ]:
# Task 1: Write your code here:

from sklearn.model_selection import KFold

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor()

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))

# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"MAE : {np.mean(mae_scores):.2f}")

In [ ]:
df_clean.columns

In [ ]:
# Task 1: Write your code here:
# Feature importance

feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Delivery Time (Ground Truth)")
plt.ylabel("Predicted Delivery Time")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Didn't have much time so left this at it is

# # Task Bonus: Write your code here:
# from catboost import CatBoostRegressor

# model1 = RandomForestRegressor()
# model2 = CatBoostRegressor()
# models = {
#   "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
#   "CatBoost": CatBoostRegressor(verbose=0)
# }

# mae_scores1 = []
# mae_scores2 = []
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
#   print(f"\nFold {fold_idx + 1}/{n_splits}")

#   X_train, X_test = X.iloc[train_index], X.iloc[test_index]
#   y_train, y_test = y.iloc[train_index], y.iloc[test_index]

#   for model_name, model in models.items():
#     print(f"Training {model_name}...")

#     # Train
#     model.fit(X_train, y_train)

#     # Predict
#     y_pred = model.predict(X_test)

#     # Calculate metrics


#     # Store results
#     all_results[model_name]["mse"].append(mse)
#     all_results[model_name]["rmse"].append(rmse)
#     all_results[model_name]["r2"].append(r2)

# # Print Evaluation Metrics
# print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
# print(f"Avg. MAE for both models: {np.mean((((mae_scores1) + np.mean(mae_scores2) / 2))):.2f}")
